In [1]:
# Data Generation Function

# Data generation
import numpy as np
from scipy.stats import bernoulli, norm

def data_simulation(sample_size_exp, sample_size_obs):
    
    # RCT
    X_trial = np.random.uniform(-2, 0.5, sample_size_exp)
    S_trial = np.ones(sample_size_exp)
    A_trial = bernoulli.rvs(0.5, size=sample_size_exp)
    tau_X_trial = 1 + X_trial + X_trial**2
    epsilon_trial = norm.rvs(0, 1.0, sample_size_exp)
    Y_trial = A_trial * tau_X_trial + X_trial**2 - 1 + epsilon_trial

    # Observational study
    X_observational = np.random.uniform(-2, 2, sample_size_obs)
    S_observational = np.zeros(sample_size_obs)
    logit_e_X_0 = X_observational
    A_observational = bernoulli.rvs(1 / (1 + np.exp(-logit_e_X_0)), size=sample_size_obs)
    tau_X_observational = 1 + X_observational + X_observational**2
    U_observational = (2 * A_observational - 1) *np.sin(X_observational-1) + norm.rvs(0, 1, size=sample_size_obs)
    epsilon_observational = norm.rvs(0, 1, size=sample_size_obs)
    Y_observational = A_observational * tau_X_observational + X_observational**2 - 1 + U_observational + epsilon_observational

    # Create DataFrames
    data_trial = np.column_stack((S_trial, X_trial, A_trial, Y_trial))
    data_observational = np.column_stack((S_observational, X_observational, A_observational, Y_observational))

    return data_trial, data_observational

In [12]:
# Model training 

import GPy
import numpy as np

def ICM(X_E, X_O, T_E, T_O, Y_E, Y_O, r, ID, AD, rho):
    """
    Train and return two rank-2 ICMs
    Args:
    - X_E (numpy.ndarray): Baseline covariate in the experimental data.
    - X_O (numpy.ndarray): Baseline covariate in the observational data.
    - T_E (numpy.ndarray): Treatment indicator in the experimental data.
    - T_O (numpy.ndarray): Treatment indicator in the observational data.
    - Y_E (numpy.ndarray): Outcome in the experimental data.
    - Y_O (numpy.ndarray): Outcome in the observational data.
    - r (int): Rank of the model.
    - ID (int): input dim
    - AD (list): active dim
    - rho: trust hyperparameter

    Returns:
    GPy.models.GPRegression: Trained Gaussian Process Regression model using Intrinsic Coregionalized Model (ICM).

    Data Processing:
    - Split the variables into treated and control for both observational and experimental data.

    Model Training:
    - Intrinsic Coregionalized Model (ICM) with a Radial Basis Function (RBF) kernel is used for regression.
    - The model is trained using Gaussian Process Regression.

    Note:
    - The order of arranging data matters: Experimental control, Observational control, Experimental treated, Observational treated.
    - An indicator is added to the baseline covariate to represent the task (0 for experimental control, 1 for observational control, 2 for experimental treated, 3 for observational treated).
    """

    # Data preprocessing
    # Split the variables in treated and control 
    # Observational Data
    X0_obs = X_O[T_O==0] # baseline covariate in th control arm of the observational study
    X1_obs = X_O[T_O==1] # baseline covariate in th treatment arm of the observational study
    Y0_obs = Y_O[T_O==0] # outcomee in th control arm of the observational study
    Y1_obs = Y_O[T_O==1] # outcomee in th treatment arm of the observational study
    
    #Exoerimental data
    X0_exp = X_E[T_E==0] # baseline covariate in th control arm of the experimental study
    X1_exp = X_E[T_E==1] # baseline covariate in th treatment arm of the experimental study
    Y0_exp = Y_E[T_E==0] # outcomee in th control arm of the experimental study
    Y1_exp = Y_E[T_E==1] # outcomee in th treatment arm of the experimental study
    

    # The order we arrange the data matters
    # 1. Experimental control
    # 2. Observational control
    # 3. Experimental treated
    # 4. Observational treated
    # We also need to add an indicator to the baseline covariate indicating the task
    # i.e. 0 is for the experiemental control, 1 is for the observational control,
    # 2 is for the experiemental treated, 3 is for observational treated
    X0_exp = np.c_[X0_exp,np.ones(X0_exp.shape[0])*0] 
    X0_obs = np.c_[X0_obs,np.ones(X0_obs.shape[0])*1]
    X1_exp = np.c_[X1_exp,np.ones(X1_exp.shape[0])*0]
    X1_obs = np.c_[X1_obs,np.ones(X1_obs.shape[0])*1]

    # Put all the data on a single array 
    Y0 = np.r_[Y0_exp, Y0_obs]
    X0 = np.r_[X0_exp, X0_obs]
    Y1 = np.r_[Y1_exp, Y1_obs]
    X1 = np.r_[X1_exp, X1_obs]


    # model training
    
    # Define the kernel
    kern = GPy.kern.RBF(input_dim=ID, active_dims = AD)**GPy.kern.Coregionalize(input_dim = 1, output_dim = 2, rank=r)
    # Define the model
    m0 = GPy.models.GPRegression(X0,np.vstack(Y0),kern)
    ##############################################################
    # Put all your constraines here
    m0.mul.coregion.W[0,0:1].fix(1)
    m0.mul.coregion.W[0,1:].fix(0)
    m0.mul.coregion.W[1,0:1].fix(rho)
    m0.mul.coregion.W[1,1:].fix(np.sqrt(1-rho**2))
    m0.mul.coregion.kappa.fix([0.000000001, 0.000000001])
    #m0.mul.rbf.variance.constrain_bounded(0., 10.)
    
    ##############################################################
    

    # Optimize the model
    m0.optimize()

    # Define the kernel
    kern = GPy.kern.RBF(input_dim=ID, active_dims = AD)**GPy.kern.Coregionalize(input_dim = 1, output_dim = 2, rank=r)
    # Define the model
    m1 = GPy.models.GPRegression(X1,np.vstack(Y1),kern)
    ##############################################################
    # Put all your constraines here
    m1.mul.coregion.W[0,0:1].fix(1)
    m1.mul.coregion.W[0,1:].fix(0)
    m1.mul.coregion.W[1,0:1].fix(rho)
    m1.mul.coregion.W[1,1:].fix(np.sqrt(1-rho**2))
    m1.mul.coregion.kappa.fix([0.000000001, 0.000000001])
    #m1.mul.rbf.variance.constrain_bounded(0., 10.)

    ##############################################################
    # Optimize the model
    
    m1.optimize()

    return m0, m1

In [13]:
import numpy as np
def weighted_mean_squared_error(weight, t, y_true, y_pred0, y_pred1):
    # Ensure inputs are numpy arrays
    y_true = np.hstack(y_true)
    y_pred0 = np.hstack(y_pred0)
    y_pred1 = np.hstack(y_pred1)
    weight = np.hstack(weight)
    weighted_squared_diff = np.zeros(y_true.shape[0])
    weighted_squared_diff[t==0] = weight[t==0]*((y_true[t==0] - y_pred0[t==0])**2) 
    weighted_squared_diff[t==1] = weight[t==1]*((y_true[t==1] - y_pred1[t==1])**2)  
    wmse = np.mean(weighted_squared_diff)
        
    return wmse

In [8]:
import numpy as np
import pandas as pd

all_data = []

rct_sample_size = 200
obs_sample_sizes = np.array([200, 500, 1000, 2000])

for i in range(50):
    for obs_n in obs_sample_sizes:
        print(i+1, obs_n, end=" ")

        data_exp, data_obs = data_simulation(
            sample_size_exp=rct_sample_size,
            sample_size_obs=obs_n
        )

        df_exp = pd.DataFrame(data_exp, columns=["S", "X", "A", "Y"])
        df_obs = pd.DataFrame(data_obs, columns=["S", "X", "A", "Y"])

        # Add metadata
        df_exp["datasetNo"] = i + 1
        df_obs["datasetNo"] = i + 1
        df_exp["rct_sample_size"] = rct_sample_size
        df_exp["obs_sample_size"] = obs_n
        df_obs["rct_sample_size"] = rct_sample_size
        df_obs["obs_sample_size"] = obs_n

        # Combine RCT + OBS
        df_all = pd.concat([df_exp, df_obs], ignore_index=True)
        all_data.append(df_all)

# Final combined object
data = pd.concat(all_data, ignore_index=True)


1 200 1 500 1 1000 1 2000 2 200 2 500 2 1000 2 2000 3 200 3 500 3 1000 3 2000 4 200 4 500 4 1000 4 2000 5 200 5 500 5 1000 5 2000 6 200 6 500 6 1000 6 2000 7 200 7 500 7 1000 7 2000 8 200 8 500 8 1000 8 2000 9 200 9 500 9 1000 9 2000 10 200 10 500 10 1000 10 2000 11 200 11 500 11 1000 11 2000 12 200 12 500 12 1000 12 2000 13 200 13 500 13 1000 13 2000 14 200 14 500 14 1000 14 2000 15 200 15 500 15 1000 15 2000 16 200 16 500 16 1000 16 2000 17 200 17 500 17 1000 17 2000 18 200 18 500 18 1000 18 2000 19 200 19 500 19 1000 19 2000 20 200 20 500 20 1000 20 2000 21 200 21 500 21 1000 21 2000 22 200 22 500 22 1000 22 2000 23 200 23 500 23 1000 23 2000 24 200 24 500 24 1000 24 2000 25 200 25 500 25 1000 25 2000 26 200 26 500 26 1000 26 2000 27 200 27 500 27 1000 27 2000 28 200 28 500 28 1000 28 2000 29 200 29 500 29 1000 29 2000 30 200 30 500 30 1000 30 2000 31 200 31 500 31 1000 31 2000 32 200 32 500 32 1000 32 2000 33 200 33 500 33 1000 33 2000 34 200 34 500 34 1000 34 2000 35 200 35 500 35

In [10]:
# save the data
data.to_csv("diff_sample_size_data.csv", index=False)


In [11]:
# Train Causal-ICM
data = pd.read_csv("diff_sample_size_data.csv")


# $n_{exp} = 200$, $n_{obs} = 200$

In [15]:
data_exp, data_obs = data_simulation(sample_size_exp=200, sample_size_obs=200)

In [16]:
# -------------------------------
# Imports and setup
# -------------------------------
import sys
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error


# -------------------------------
# Set seeds for reproducibility
# -------------------------------
seed_value = 1234
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)


data_full = np.r_[data_exp, data_obs]

# -------------------------------
# Logistic regression for trial participation
# -------------------------------
from sklearn import linear_model
logr = linear_model.LogisticRegression()
logr.fit(data_full[:,1].reshape(-1,1), data_full[:,0])
prob = logr.predict_proba(data_full[:,1].reshape(-1,1))
prob_part = prob[:,1]
data_full = np.c_[data_full, 1/prob_part]


# -------------------------------
# Rho tuning via cross-validation
# -------------------------------
from sklearn.model_selection import KFold

rho_values = np.round(np.arange(0, 1.05, 0.1), 1)
num_folds = 5
min_avg_wmse = float('inf')
best_rho = None
wmse_per_rho = {}

kf = KFold(n_splits=num_folds, shuffle=True)
for rho in rho_values:
    print(f"Rho: {rho}")
    avg_wmse = 0
    fold_num = 0
    wmse_per_fold = []
    for train_index, val_index in kf.split(data_full):
        train_data, val_data = data_full[train_index], data_full[val_index]
        val_data = val_data[val_data[:,0]==1]
        X_val, T_val, Y_val, w = val_data[:,1], val_data[:,2], val_data[:,3], val_data[:,4]

        m0, m1 = ICM(
            X_E=train_data[train_data[:,0]==1,1],
            X_O=train_data[train_data[:,0]==0,1],
            T_E=train_data[train_data[:,0]==1,2],
            T_O=train_data[train_data[:,0]==0,2],
            Y_E=train_data[train_data[:,0]==1,3],
            Y_O=train_data[train_data[:,0]==0,3],
            r=2, ID=1, AD=0, rho=rho
        )

        predY0_exp = m0.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        predY1_exp = m1.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        wmse = weighted_mean_squared_error(weight=w, t=T_val, y_true=Y_val, y_pred0=predY0_exp[0], y_pred1=predY1_exp[0])
        avg_wmse += wmse
        wmse_per_fold.append(wmse)

        print(f"Fold {fold_num + 1} WMSE: {wmse}")
        fold_num += 1

    avg_wmse /= num_folds
    print(f"Average WMSE for Rho {rho}: {avg_wmse}")
    wmse_per_rho[rho] = wmse_per_fold
    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho

print(f"Best Rho: {best_rho}, Minimum Average WMSE: {min_avg_wmse}")
pd.DataFrame([best_rho], columns=['best_rho']).to_csv('sim8_best_rho.csv', index=False)



Rho: 0.0
Fold 1 WMSE: 2.0064512121277045
Fold 2 WMSE: 1.3801214548173033
Fold 3 WMSE: 2.0693770639943225
Fold 4 WMSE: 1.9275304300904414
Fold 5 WMSE: 2.0573209153487535
Average WMSE for Rho 0.0: 1.888160215275705
Rho: 0.1
Fold 1 WMSE: 1.699865825356539
Fold 2 WMSE: 2.006252497998645
Fold 3 WMSE: 2.7482695860088193
Fold 4 WMSE: 1.4559434471305321
Fold 5 WMSE: 1.5942839580032924
Average WMSE for Rho 0.1: 1.9009230628995657
Rho: 0.2
Fold 1 WMSE: 2.0180279624802777
Fold 2 WMSE: 1.8609315159051205
Fold 3 WMSE: 2.348002908763171
Fold 4 WMSE: 1.9800772879686948
Fold 5 WMSE: 1.6626661637557891
Average WMSE for Rho 0.2: 1.9739411677746108
Rho: 0.3
Fold 1 WMSE: 1.6366716710097158
Fold 2 WMSE: 2.5565009463461736
Fold 3 WMSE: 1.7754090828934366
Fold 4 WMSE: 3.086050186189979
Fold 5 WMSE: 1.0762762316442849
Average WMSE for Rho 0.3: 2.026181623616718
Rho: 0.4
Fold 1 WMSE: 2.5988869295510537
Fold 2 WMSE: 1.0373165339151984
Fold 3 WMSE: 2.3103655629076765
Fold 4 WMSE: 1.8526521477978266
Fold 5 WMSE: 

In [19]:
best_rho_200 = best_rho
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2)
trueCATE = 1+test_data+test_data**2

In [38]:
rmse_nobs200 = np.zeros((50))
bias_nobs200 = np.zeros((50))
var_nobs200 = np.zeros((50))

for i in range(50):
    print(i + 1, end=" ")
    data_exp = data[(data["S"]==1) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==200)]
    data_obs = data[(data["S"]==0) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==200)]


    m0, m1 = ICM(X_E = np.hstack(data_exp["X"].values), X_O = np.hstack(data_obs["X"].values), 
                 T_E = np.hstack(data_exp["A"].values), T_O = np.hstack(data_obs["A"].values), 
                 Y_E = np.hstack(data_exp["Y"].values), Y_O = np.hstack(data_obs["Y"].values),  
                 r = 2, ID=1, AD=0, rho=best_rho_200)
    predY0 = m0.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predY1 = m1.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predCATE = predY1[0] - predY0[0]

    

    rmse_nobs200[i] = np.sqrt(mean_squared_error((predCATE), (trueCATE)))
    bias_nobs200[i] = np.mean(predCATE - trueCATE)
    var_nobs200[i] = np.mean((predCATE - np.mean(predCATE))**2)

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 

# $n_{exp} = 200$, $n_{obs} = 500$

In [39]:
# -------------------------------
# Imports and setup
# -------------------------------
import sys
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error

data_exp, data_obs = data_simulation(sample_size_exp=200, sample_size_obs=500)
# -------------------------------
# Set seeds for reproducibility
# -------------------------------
seed_value = 1234
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)


data_full = np.r_[data_exp, data_obs]

# -------------------------------
# Logistic regression for trial participation
# -------------------------------
from sklearn import linear_model
logr = linear_model.LogisticRegression()
logr.fit(data_full[:,1].reshape(-1,1), data_full[:,0])
prob = logr.predict_proba(data_full[:,1].reshape(-1,1))
prob_part = prob[:,1]
data_full = np.c_[data_full, 1/prob_part]


# -------------------------------
# Rho tuning via cross-validation
# -------------------------------
from sklearn.model_selection import KFold

rho_values = np.round(np.arange(0, 1.05, 0.1), 1)
num_folds = 5
min_avg_wmse = float('inf')
best_rho = None
wmse_per_rho = {}

kf = KFold(n_splits=num_folds, shuffle=True)
for rho in rho_values:
    print(f"Rho: {rho}")
    avg_wmse = 0
    fold_num = 0
    wmse_per_fold = []
    for train_index, val_index in kf.split(data_full):
        train_data, val_data = data_full[train_index], data_full[val_index]
        val_data = val_data[val_data[:,0]==1]
        X_val, T_val, Y_val, w = val_data[:,1], val_data[:,2], val_data[:,3], val_data[:,4]

        m0, m1 = ICM(
            X_E=train_data[train_data[:,0]==1,1],
            X_O=train_data[train_data[:,0]==0,1],
            T_E=train_data[train_data[:,0]==1,2],
            T_O=train_data[train_data[:,0]==0,2],
            Y_E=train_data[train_data[:,0]==1,3],
            Y_O=train_data[train_data[:,0]==0,3],
            r=2, ID=1, AD=0, rho=rho
        )

        predY0_exp = m0.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        predY1_exp = m1.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        wmse = weighted_mean_squared_error(weight=w, t=T_val, y_true=Y_val, y_pred0=predY0_exp[0], y_pred1=predY1_exp[0])
        avg_wmse += wmse
        wmse_per_fold.append(wmse)

        print(f"Fold {fold_num + 1} WMSE: {wmse}")
        fold_num += 1

    avg_wmse /= num_folds
    print(f"Average WMSE for Rho {rho}: {avg_wmse}")
    wmse_per_rho[rho] = wmse_per_fold
    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho

print(f"Best Rho: {best_rho}, Minimum Average WMSE: {min_avg_wmse}")
pd.DataFrame([best_rho], columns=['best_rho']).to_csv('sim8_best_rho.csv', index=False)


best_rho_500 = best_rho
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2)
trueCATE = 1+test_data+test_data**2
rmse_nobs500 = np.zeros((50))
bias_nobs500 = np.zeros((50))
var_nobs500 = np.zeros((50))

for i in range(50):
    print(i + 1, end=" ")
    data_exp = data[(data["S"]==1) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==500)]
    data_obs = data[(data["S"]==0) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==500)]


    m0, m1 = ICM(X_E = np.hstack(data_exp["X"].values), X_O = np.hstack(data_obs["X"].values), 
                 T_E = np.hstack(data_exp["A"].values), T_O = np.hstack(data_obs["A"].values), 
                 Y_E = np.hstack(data_exp["Y"].values), Y_O = np.hstack(data_obs["Y"].values),  
                 r = 2, ID=1, AD=0, rho=best_rho_200)
    predY0 = m0.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predY1 = m1.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predCATE = predY1[0] - predY0[0]

    

    rmse_nobs500[i] = np.sqrt(mean_squared_error((predCATE), (trueCATE)))
    bias_nobs500[i] = np.mean(predCATE - trueCATE)
    var_nobs500[i] = np.mean((predCATE - np.mean(predCATE))**2)

Rho: 0.0
Fold 1 WMSE: 1.6043252892179727
Fold 2 WMSE: 3.305331537189552
Fold 3 WMSE: 3.1439263607355414
Fold 4 WMSE: 3.0676705383916234
Fold 5 WMSE: 2.752106285715287
Average WMSE for Rho 0.0: 2.7746720022499956
Rho: 0.1
Fold 1 WMSE: 2.308720122599409
Fold 2 WMSE: 2.4152632995657353
Fold 3 WMSE: 3.5966119492658946
Fold 4 WMSE: 3.305609174235068
Fold 5 WMSE: 2.0529191282894055
Average WMSE for Rho 0.1: 2.7358247347911027
Rho: 0.2
Fold 1 WMSE: 2.571849284768552
Fold 2 WMSE: 2.6013999258054747
Fold 3 WMSE: 3.2338001949805637
Fold 4 WMSE: 2.716392806335366
Fold 5 WMSE: 2.3030545014627
Average WMSE for Rho 0.2: 2.6852993426705316
Rho: 0.3
Fold 1 WMSE: 2.0180338434708167
Fold 2 WMSE: 2.7816496212663324
Fold 3 WMSE: 3.0001338307570853
Fold 4 WMSE: 3.1915845086768875
Fold 5 WMSE: 3.0860384250971293
Average WMSE for Rho 0.3: 2.8154880458536504
Rho: 0.4
Fold 1 WMSE: 2.365324554053559
Fold 2 WMSE: 3.2560373591352225
Fold 3 WMSE: 2.906063844325315
Fold 4 WMSE: 2.57490469898079
Fold 5 WMSE: 2.33160

In [40]:
# -------------------------------
# Imports and setup
# -------------------------------
import sys
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error

data_exp, data_obs = data_simulation(sample_size_exp=200, sample_size_obs=1000)
# -------------------------------
# Set seeds for reproducibility
# -------------------------------
seed_value = 1234
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)


data_full = np.r_[data_exp, data_obs]

# -------------------------------
# Logistic regression for trial participation
# -------------------------------
from sklearn import linear_model
logr = linear_model.LogisticRegression()
logr.fit(data_full[:,1].reshape(-1,1), data_full[:,0])
prob = logr.predict_proba(data_full[:,1].reshape(-1,1))
prob_part = prob[:,1]
data_full = np.c_[data_full, 1/prob_part]


# -------------------------------
# Rho tuning via cross-validation
# -------------------------------
from sklearn.model_selection import KFold

rho_values = np.round(np.arange(0, 1.05, 0.1), 1)
num_folds = 5
min_avg_wmse = float('inf')
best_rho = None
wmse_per_rho = {}

kf = KFold(n_splits=num_folds, shuffle=True)
for rho in rho_values:
    print(f"Rho: {rho}")
    avg_wmse = 0
    fold_num = 0
    wmse_per_fold = []
    for train_index, val_index in kf.split(data_full):
        train_data, val_data = data_full[train_index], data_full[val_index]
        val_data = val_data[val_data[:,0]==1]
        X_val, T_val, Y_val, w = val_data[:,1], val_data[:,2], val_data[:,3], val_data[:,4]

        m0, m1 = ICM(
            X_E=train_data[train_data[:,0]==1,1],
            X_O=train_data[train_data[:,0]==0,1],
            T_E=train_data[train_data[:,0]==1,2],
            T_O=train_data[train_data[:,0]==0,2],
            Y_E=train_data[train_data[:,0]==1,3],
            Y_O=train_data[train_data[:,0]==0,3],
            r=2, ID=1, AD=0, rho=rho
        )

        predY0_exp = m0.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        predY1_exp = m1.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        wmse = weighted_mean_squared_error(weight=w, t=T_val, y_true=Y_val, y_pred0=predY0_exp[0], y_pred1=predY1_exp[0])
        avg_wmse += wmse
        wmse_per_fold.append(wmse)

        print(f"Fold {fold_num + 1} WMSE: {wmse}")
        fold_num += 1

    avg_wmse /= num_folds
    print(f"Average WMSE for Rho {rho}: {avg_wmse}")
    wmse_per_rho[rho] = wmse_per_fold
    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho

print(f"Best Rho: {best_rho}, Minimum Average WMSE: {min_avg_wmse}")
pd.DataFrame([best_rho], columns=['best_rho']).to_csv('sim8_best_rho.csv', index=False)


best_rho_1000 = best_rho
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2)
trueCATE = 1+test_data+test_data**2
rmse_nobs1000 = np.zeros((50))
bias_nobs1000 = np.zeros((50))
var_nobs1000 = np.zeros((50))

for i in range(50):
    print(i + 1, end=" ")
    data_exp = data[(data["S"]==1) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==1000)]
    data_obs = data[(data["S"]==0) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==1000)]


    m0, m1 = ICM(X_E = np.hstack(data_exp["X"].values), X_O = np.hstack(data_obs["X"].values), 
                 T_E = np.hstack(data_exp["A"].values), T_O = np.hstack(data_obs["A"].values), 
                 Y_E = np.hstack(data_exp["Y"].values), Y_O = np.hstack(data_obs["Y"].values),  
                 r = 2, ID=1, AD=0, rho=best_rho_200)
    predY0 = m0.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predY1 = m1.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predCATE = predY1[0] - predY0[0]

    

    rmse_nobs1000[i] = np.sqrt(mean_squared_error((predCATE), (trueCATE)))
    bias_nobs1000[i] = np.mean(predCATE - trueCATE)
    var_nobs1000[i] = np.mean((predCATE - np.mean(predCATE))**2)

Rho: 0.0
Fold 1 WMSE: 5.357253803820309
Fold 2 WMSE: 5.914250946626653
Fold 3 WMSE: 6.731583192263785
Fold 4 WMSE: 7.712537065823465
Fold 5 WMSE: 4.982646641366893
Average WMSE for Rho 0.0: 6.13965432998022
Rho: 0.1
Fold 1 WMSE: 6.085521754917561
Fold 2 WMSE: 4.593868331094801
Fold 3 WMSE: 6.157982378156396
Fold 4 WMSE: 7.4074639088957
Fold 5 WMSE: 7.152443749571122
Average WMSE for Rho 0.1: 6.279456024527116
Rho: 0.2
Fold 1 WMSE: 6.5434548046045675
Fold 2 WMSE: 5.975548620482252
Fold 3 WMSE: 5.8638499739063565
Fold 4 WMSE: 7.324636977495167
Fold 5 WMSE: 5.942292011830734
Average WMSE for Rho 0.2: 6.329956477663815
Rho: 0.3
Fold 1 WMSE: 7.580802650307526
Fold 2 WMSE: 5.314061336545719
Fold 3 WMSE: 6.820681132900383
Fold 4 WMSE: 5.299741626088681
Fold 5 WMSE: 6.858200133737979
Average WMSE for Rho 0.3: 6.374697375916059
Rho: 0.4
Fold 1 WMSE: 6.740653724092996
Fold 2 WMSE: 6.527774465678081
Fold 3 WMSE: 6.212769517506951
Fold 4 WMSE: 4.949141329051618
Fold 5 WMSE: 7.390290872665792
Avera

In [41]:
# -------------------------------
# Imports and setup
# -------------------------------
import sys
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error

data_exp, data_obs = data_simulation(sample_size_exp=200, sample_size_obs=2000)
# -------------------------------
# Set seeds for reproducibility
# -------------------------------
seed_value = 1234
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)


data_full = np.r_[data_exp, data_obs]

# -------------------------------
# Logistic regression for trial participation
# -------------------------------
from sklearn import linear_model
logr = linear_model.LogisticRegression()
logr.fit(data_full[:,1].reshape(-1,1), data_full[:,0])
prob = logr.predict_proba(data_full[:,1].reshape(-1,1))
prob_part = prob[:,1]
data_full = np.c_[data_full, 1/prob_part]


# -------------------------------
# Rho tuning via cross-validation
# -------------------------------
from sklearn.model_selection import KFold

rho_values = np.round(np.arange(0, 1.05, 0.1), 1)
num_folds = 5
min_avg_wmse = float('inf')
best_rho = None
wmse_per_rho = {}

kf = KFold(n_splits=num_folds, shuffle=True)
for rho in rho_values:
    print(f"Rho: {rho}")
    avg_wmse = 0
    fold_num = 0
    wmse_per_fold = []
    for train_index, val_index in kf.split(data_full):
        train_data, val_data = data_full[train_index], data_full[val_index]
        val_data = val_data[val_data[:,0]==1]
        X_val, T_val, Y_val, w = val_data[:,1], val_data[:,2], val_data[:,3], val_data[:,4]

        m0, m1 = ICM(
            X_E=train_data[train_data[:,0]==1,1],
            X_O=train_data[train_data[:,0]==0,1],
            T_E=train_data[train_data[:,0]==1,2],
            T_O=train_data[train_data[:,0]==0,2],
            Y_E=train_data[train_data[:,0]==1,3],
            Y_O=train_data[train_data[:,0]==0,3],
            r=2, ID=1, AD=0, rho=rho
        )

        predY0_exp = m0.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        predY1_exp = m1.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape)])
        wmse = weighted_mean_squared_error(weight=w, t=T_val, y_true=Y_val, y_pred0=predY0_exp[0], y_pred1=predY1_exp[0])
        avg_wmse += wmse
        wmse_per_fold.append(wmse)

        print(f"Fold {fold_num + 1} WMSE: {wmse}")
        fold_num += 1

    avg_wmse /= num_folds
    print(f"Average WMSE for Rho {rho}: {avg_wmse}")
    wmse_per_rho[rho] = wmse_per_fold
    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho

print(f"Best Rho: {best_rho}, Minimum Average WMSE: {min_avg_wmse}")
pd.DataFrame([best_rho], columns=['best_rho']).to_csv('sim8_best_rho.csv', index=False)


best_rho_2000 = best_rho
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2)
trueCATE = 1+test_data+test_data**2
rmse_nobs2000 = np.zeros((50))
bias_nobs2000 = np.zeros((50))
var_nobs2000 = np.zeros((50))

for i in range(50):
    print(i + 1, end=" ")
    data_exp = data[(data["S"]==1) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==2000)]
    data_obs = data[(data["S"]==0) & (data["datasetNo"]==i+1) & (data["obs_sample_size"]==2000)]


    m0, m1 = ICM(X_E = np.hstack(data_exp["X"].values), X_O = np.hstack(data_obs["X"].values), 
                 T_E = np.hstack(data_exp["A"].values), T_O = np.hstack(data_obs["A"].values), 
                 Y_E = np.hstack(data_exp["Y"].values), Y_O = np.hstack(data_obs["Y"].values),  
                 r = 2, ID=1, AD=0, rho=best_rho_200)
    predY0 = m0.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predY1 = m1.predict_noiseless(np.c_[np.vstack(test_data),np.ones(test_data.shape[0])*0])
    predCATE = predY1[0] - predY0[0]

    

    rmse_nobs2000[i] = np.sqrt(mean_squared_error((predCATE), (trueCATE)))
    bias_nobs2000[i] = np.mean(predCATE - trueCATE)
    var_nobs2000[i] = np.mean((predCATE - np.mean(predCATE))**2)

Rho: 0.0
Fold 1 WMSE: 15.98159764291428
Fold 2 WMSE: 11.350587103731447
Fold 3 WMSE: 14.292284057817447
Fold 4 WMSE: 11.21274124374835
Fold 5 WMSE: 8.243735160380089
Average WMSE for Rho 0.0: 12.216189041718325
Rho: 0.1
Fold 1 WMSE: 11.67104311907311
Fold 2 WMSE: 16.491416672250605
Fold 3 WMSE: 11.56295967578205
Fold 4 WMSE: 9.80669951770345
Fold 5 WMSE: 12.530828080421658
Average WMSE for Rho 0.1: 12.412589413046174
Rho: 0.2
Fold 1 WMSE: 12.956265417042498
Fold 2 WMSE: 8.980391167037316
Fold 3 WMSE: 14.787182077536325
Fold 4 WMSE: 14.655580366364555
Fold 5 WMSE: 9.53964262322536
Average WMSE for Rho 0.2: 12.183812330241212
Rho: 0.3
Fold 1 WMSE: 9.55806845827253
Fold 2 WMSE: 5.3395545679875696
Fold 3 WMSE: 12.15519207084602
Fold 4 WMSE: 14.576885974531928
Fold 5 WMSE: 18.595429552242592
Average WMSE for Rho 0.3: 12.045026124776127
Rho: 0.4
Fold 1 WMSE: 13.07118218161481
Fold 2 WMSE: 13.0506901582816
Fold 3 WMSE: 12.865730922600877
Fold 4 WMSE: 11.939674415579363
Fold 5 WMSE: 10.3272764

In [46]:
results_df = pd.concat([
    pd.DataFrame({
        "obs_sample_size": 200,
        "rep": np.arange(1, 51),
        "best_rho": best_rho_200,
        "rmse": rmse_nobs200,
        "bias": bias_nobs200,
        "variance": var_nobs200
    }),
    pd.DataFrame({
        "obs_sample_size": 500,
        "rep": np.arange(1, 51),
        "best_rho": best_rho_500,
        "rmse": rmse_nobs500,
        "bias": bias_nobs500,
        "variance": var_nobs500
    }),
    pd.DataFrame({
        "obs_sample_size": 1000,
        "rep": np.arange(1, 51),
        "best_rho": best_rho_1000,
        "rmse": rmse_nobs1000,
        "bias": bias_nobs1000,
        "variance": var_nobs1000
    }),
    pd.DataFrame({
        "obs_sample_size": 2000,
        "rep": np.arange(1, 51),
        "best_rho": best_rho_2000,
        "rmse": rmse_nobs2000,
        "bias": bias_nobs2000,
        "variance": var_nobs2000
    })
], ignore_index=True)


In [48]:
results_df.to_csv("sample_size_imbalance_results.csv", index=False)


In [49]:
mean_rmse_df = (
    results_df
    .groupby("obs_sample_size", as_index=False)["rmse"]
    .mean()
    .rename(columns={"rmse": "mean_rmse"})
)

print(mean_rmse_df)


   obs_sample_size  mean_rmse
0              200   1.078376
1              500   1.071102
2             1000   0.928559
3             2000   0.933797
